# Revalidation technique de l'alerte comportementale - MemoireV3

Ce notebook reproduit le noyau de validation technique du mémoire V3: campagne post-baseline, revue humaine guidée, synthèses par scénario, ablation des comparateurs, benchmark et manifeste de non-régression. Il ne diagnostique pas la boiterie, ne constitue pas une validation clinique et conserve IF/LOF uniquement comme comparateurs techniques.


In [ ]:
import json
import os
import sys
from dataclasses import asdict
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd  # noqa: E402

from core.early_warning import EarlyWarningConfig  # noqa: E402
from revalidation import analysis as A  # noqa: E402
from revalidation.ablation import ablation_paired_tests, ablation_summary, run_clean_ablation  # noqa: E402
from revalidation.campaign import run_clean_campaign  # noqa: E402
from revalidation.qa import assert_campaign_valid, manual_review_sample, write_json, write_sha256_manifest  # noqa: E402

# Parametres figes pour reproductibilite
PINNED_PARAMS = dict(
    interval='15T',
    window_baseline=24,
    contamination=0.06,
    baseline_ratio=0.60,
    random_state=42,
    persist_hours=7,
    alert_min=2,
    mix_mode='MIX',
    mix_rate_thr=0.24,
    z_low_thr=-2.0,
    z_high_thr=2.0,
    cooldown_hours=12,
    mi_z_high_thr=2.2,
    coverage_min_pct=25.0,
)
PINNED_WARNING_CONFIG = EarlyWarningConfig(
    aggregation_hours=12.0,
    persistence_hours=6.0,
    cooldown_hours=24.0,
    family_min_change=0.10,
    posture_min_change=0.05,
    score_threshold=0.12,
    cusum_drift=0.070,
    cusum_threshold=1.20,
    min_families=3,
    coverage_min_pct=25.0,
)
PRIMARY_SEEDS = [11]
OUTPUT = ROOT / 'data' / 'revalidation'
OUTPUT.mkdir(parents=True, exist_ok=True)
for obsolete in ['calibration_grouped.json']:
    (OUTPUT / obsolete).unlink(missing_ok=True)
write_json(
    OUTPUT / 'protocol_configuration.json',
    {
        'pipeline': PINNED_PARAMS,
        'early_warning': asdict(PINNED_WARNING_CONFIG),
        'primary_seeds': PRIMARY_SEEDS,
        'scope': 'benchmark technique post-baseline a configuration documentee',
    },
)
print('Racine:', ROOT)
print('Sorties:', OUTPUT)
print('Configuration pipeline:', PINNED_PARAMS)
print('Configuration alerte:', asdict(PINNED_WARNING_CONFIG))

## 1. Smoke test bloquant

Ce test réduit vérifie l'exécution et les invariants du protocole avant la campagne complète.

In [ ]:
smoke = run_clean_campaign(
    raw_csv='data/brut.csv',
    cows=['8081', '8147', '2056'],
    seeds=PRIMARY_SEEDS,
    params=PINNED_PARAMS,
    warning_config=PINNED_WARNING_CONFIG,
    verbose=True,
)
smoke_checks = assert_campaign_valid(smoke)
display(smoke_checks)
display(A.detection_summary(smoke))

## 2. Campagne primaire post-baseline

Une seule seed primaire est utilisée. Un seul événement est injecté par exécution, uniquement sur les vaches disposant d'au moins 14 jours de données et de signaux d'activité informatifs dans la période post-baseline. Les événements d'une même vache restent corrélés; toute inférence est regroupée par vache.

In [ ]:
events = run_clean_campaign(
    raw_csv='data/brut.csv',
    seeds=PRIMARY_SEEDS,
    params=PINNED_PARAMS,
    warning_config=PINNED_WARNING_CONFIG,
    verbose=True,
)
checks = assert_campaign_valid(events)
events_path = OUTPUT / 'events_primary.csv'
checks_path = OUTPUT / 'protocol_checks.csv'
events.to_csv(events_path, index=False)
checks.to_csv(checks_path, index=False)
display(checks)
print('Événements:', len(events), '| vaches:', events['cow'].nunique())

## 3. Revue humaine guidée

Le fichier produit sélectionne notamment les cas les moins bien détectés. Compléter `manual_status` et `manual_comment`; cette étape n'est pas automatisable.

In [ ]:
review = manual_review_sample(events, per_scenario=3)
review_path = OUTPUT / 'manual_review_sample.csv'
annotation_path = OUTPUT / 'manual_review_annotations.csv'
if annotation_path.exists():
    annotations = pd.read_csv(annotation_path)
    review = review.drop(columns=['manual_status', 'manual_comment']).merge(
        annotations[['event_id', 'manual_status', 'manual_comment']],
        on='event_id',
        how='left',
    )
    review['manual_status'] = review['manual_status'].fillna('A_REVOIR')
    review['manual_comment'] = review['manual_comment'].fillna('')
review.to_csv(review_path, index=False)
display(review)
print('Revue manuelle:', review_path)

## 4. Couverture technique et discrimination synthétique par vache

In [ ]:
detection = A.detection_summary(events)
detection.to_csv(OUTPUT / 'detection_summary.csv', index=False)
display(detection)
print('Alertes de fond non adjudicables:', A.background_notification_rate(events))
print('Alerte nouvelle sur contrôle bref:', A.leak_rate(events))


## 5. Analyse de sensibilité par niveau synthétique

Les niveaux servent à vérifier la réponse du système à plusieurs amplitudes. Le score n'est pas interprété comme une échelle clinique ordinale.

In [ ]:
intensity = (
    events.groupby('scenario', as_index=False)
    .agg(
        n_events=('event_id', 'nunique'),
        new_warning_rate=('detected_any_overlap', 'mean'),
        coverage_rate=('episode_overlap', 'mean'),
        warning_score_median=('warning_score_max', 'median'),
        warning_score_q25=('warning_score_max', lambda x: x.quantile(0.25)),
        warning_score_q75=('warning_score_max', lambda x: x.quantile(0.75)),
    )
)
intensity.to_csv(OUTPUT / 'intensity_sensitivity.csv', index=False)
display(intensity.round(3))

## 6. Comparaison post-baseline des détecteurs

Le détecteur temporel est comparé à IF avec règles de persistance, à IF ponctuel et à LOF avec règles. IF et LOF utilisent les mêmes points de baseline; les tests appariés utilisent les moyennes par vache.

In [ ]:
ablation = run_clean_ablation(
    raw_csv='data/brut.csv',
    seeds=PRIMARY_SEEDS,
    params=PINNED_PARAMS,
    warning_config=PINNED_WARNING_CONFIG,
    verbose=True,
)
ablation_raw_path = OUTPUT / 'ablation_primary.csv'
ablation_summary_path = OUTPUT / 'ablation_summary.csv'
ablation_tests_path = OUTPUT / 'ablation_tests_by_cow.csv'
ablation.to_csv(ablation_raw_path, index=False)
ablation_summary(ablation).to_csv(ablation_summary_path, index=False)
ablation_paired_tests(ablation).to_csv(ablation_tests_path, index=False)
display(ablation_summary(ablation))
display(ablation_paired_tests(ablation))

## 7. Benchmark de performance

Un passage sur tout le troupeau vérifie que la configuration finale reste exécutable à l'échelle du corpus.

In [ ]:
from core.io import load_csv
from scripts.benchmark_performance import PipelineParams, run_benchmark_herd

performance_params = PipelineParams(**PINNED_PARAMS)
performance = run_benchmark_herd(
    load_csv('data/brut.csv'),
    performance_params,
    collect_output=False,
)
write_json(OUTPUT / 'performance_full_corpus.json', performance)
print(json.dumps(performance, indent=2, ensure_ascii=False))

## 8. Manifeste de non-régression

In [ ]:
artifact_paths = [
    path for path in OUTPUT.rglob('*')
    if path.is_file() and path.name != 'revalidation.sha256'
    and path.suffix.lower() in {'.csv', '.json', '.png'}
]
manifest = write_sha256_manifest(artifact_paths, OUTPUT / 'revalidation.sha256')
print('Manifeste:', manifest)
print(manifest.read_text(encoding='utf-8'))

## 9. Règle de mise à jour du mémoire

Ne reporter les résultats qu'après: (1) succès de tous les contrôles, (2) revue humaine complétée, (3) conservation du notebook exécuté et du manifeste, et (4) formulation explicite en **validation technique synthétique d'une alerte comportementale**. Sans scores locomoteurs datés ou diagnostic vétérinaire, aucun taux de sensibilité clinique ne peut être affirmé.